<img src="../images/bwHPC_Logo_cmyk.svg" width="200" /> <img src="../images/HochschuleEsslingen_Logo_RGB_DE.png" width="200" /> <img src="../images/Konstanz_Logo.svg" width="200" /> <img src="../images/KIT_Logo.png" width="200" />

# Dask on the Cluster

> ### Please note: Every command in this notebook is written for the bwUniCluster.
>
> The partition
> name, the node sizes, the module that provides MPI, the network interface, the
> `$TMP` directory and the `sinfo_t_idle` helper are all specific to it.
>
> The notebook is still useful on another HPC system, but it is not portable as it
> stands: you will have to adapt values as you go. [Section 3](#sizing) collects the
> site-specific settings into a single cell to make that as painless as possible, and
> every later section says which of them it depends on. If your cluster uses a
> scheduler other than SLURM, the job submission in
> [sections 4 and 5](#jobscript) has to be rewritten entirely.

[`01_Introduction_To_Dask.ipynb`](01_Introduction_To_Dask.ipynb) covered what Dask is,
what a Dask Array and a Dask DataFrame are made of, and how lazy evaluation and
`.compute()` work. All of that is assumed here and none of it is repeated.

This notebook changes exactly one thing: **where the workers live.** In the first
notebook they were four threads inside the Jupyter process. Here they become
processes spread over several compute nodes, reserved through SLURM and started with
`dask-mpi`. The Dask code you write afterwards does not change at all — which is the
whole point, and worth keeping in mind while working through the setup below, because
the setup is the only genuinely new material.

---

## Contents

1. [From one node to many](#many)
2. [The environment on the compute nodes](#environment)
3. [Sizing the job](#sizing)
4. [The job script](#jobscript)
5. [Submitting and monitoring the job](#submitting)
6. [Connecting to the cluster](#connecting)
7. [Computing at cluster scale](#computing)
8. [Growing the cluster while it runs](#growing)
9. [Machine learning on the cluster](#ml)
10. [Freeing the resources](#freeing)

<a id="many"></a>
## 1. From one node to many

A Dask cluster is a scheduler plus some number of workers. Creating a `Client()` the
way the first notebook did starts both on the spot, inside your Jupyter session, and
that limits you to the one node your JupyterLab job is running on.

To use more than one node, the nodes have to be reserved from SLURM first, and the
scheduler and workers started on them. There are two ways to do that, and the
difference matters enough to be worth a paragraph each.

**`dask_jobqueue`** submits one SLURM job per node, and grows or shrinks the cluster
by submitting and cancelling jobs. That model fits clusters where a job means a node.
On the bwUniCluster it fits badly, for two reasons. Each job is scheduled
independently, so it is only by luck that all of them start at the same time — and
until they do, your cluster is incomplete. On top of that, the `cpu` queue allows
several nodes per job, and `dask_jobqueue` will happily reserve a job of several nodes
and then use only one of them.

**`dask-mpi`** starts the whole cluster with one `mpirun` inside a *single* SLURM job.
One job means one start time, so either every node is there or none is. It also means
a queue that hands out several nodes per job is used the way it was intended.

For those two reasons everything below uses `dask-mpi`. `dask_jobqueue` is still worth
recognising, so here is what it looks like — as a **raw cell**, so that "Run All
Cells" cannot accidentally submit it:

The queues available on the bwUniCluster, and how many nodes each one allows per job,
are documented at
[https://wiki.bwhpc.de/e/BwUniCluster3.0/Running_Jobs#Queues_on_bwUniCluster_3.0](https://wiki.bwhpc.de/e/BwUniCluster3.0/Running_Jobs#Queues_on_bwUniCluster_3.0).

<a id="environment"></a>
## 2. The environment on the compute nodes

Every worker is a Python process, so every node needs the workshop's packages. Reading
them from `$HOME` over the shared filesystem works, but a Python import touches
thousands of small files, and doing that from every node at once is exactly the access
pattern a network filesystem is worst at.

So each node builds its own copy on its **local disk** instead. `uv` makes this
straightforward: pointing `UV_PROJECT_ENVIRONMENT` at a directory under `$TMP` and
running `uv sync` there produces a complete environment in seconds, from the package
cache in `$HOME` that your `uv sync --all-extras` on the login node already filled.

Three things about that are worth knowing before the job script in
[section 4](#jobscript) uses them:

- **Copying `.venv` from `$HOME` would not work.** `uv` writes absolute paths into the
  console scripts of an environment and symlinks `bin/python` to the interpreter in
  `~/.local/share/uv/python/`. A copy of `.venv` therefore still points back at
  `$HOME`. Running `uv sync` *at* the target directory is what gets the paths right.
- **Compute nodes have no route to the internet.** `UV_OFFLINE=1` makes `uv` work from
  the local cache only and fail loudly instead of hanging on a download. This is why
  the cache has to be warm before the job starts.
- **Only the extras the job needs are installed.** `--extra dask --extra mpi` leaves
  out TensorFlow and PyTorch, which are several GB that no Dask worker will ever
  import.

First, the pieces the rest of the notebook needs.

In [ ]:
import os
import pathlib
import re
import shutil
import subprocess
import time


def shell(command):
    """Run a shell command and return its output as a string.

    os.system() sends output to the log of the JupyterLab job rather than into the
    notebook, so a cell calling it appears to print nothing but a return code.
    Every shell command in this notebook goes through this function instead.
    """
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    return (result.stdout + result.stderr).rstrip()


def find_repo_root(start=None):
    """Walk upwards from `start` until the directory holding pyproject.toml is found."""
    path = pathlib.Path(start or os.getcwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("pyproject.toml not found -- is this a clone of the workshop?")


repo_root = find_repo_root()
uv_bin = shutil.which("uv") or str(pathlib.Path.home() / ".local" / "bin" / "uv")

print("repository:", repo_root)
print("uv binary: ", uv_bin, "-- found:", pathlib.Path(uv_bin).exists())

The absolute `repo_root` matters more than it looks. The job script and the workers run
on other nodes, in whatever directory SLURM gives them, so a relative path like
`../files/...` would be resolved against the wrong place. Everything below is built
from `repo_root`.

Now make sure the cache is warm. This is a no-op if you already ran
`uv sync --all-extras` as the README describes, and it fills the cache if you did not.

In [ ]:
print(shell(f"cd {repo_root} && {uv_bin} sync --all-extras"))

<a id="sizing"></a>
## 3. Sizing the job

This is the cell to edit, and the only one that describes the machine. The hardware
figures come from
[https://wiki.bwhpc.de/e/BwUniCluster3.0/Hardware_and_Architecture](https://wiki.bwhpc.de/e/BwUniCluster3.0/Hardware_and_Architecture).

The one decision with real consequences is how to split each node's cores between
**processes** and **threads**, and the rule follows from Python's Global Interpreter
Lock. Numerical libraries — NumPy, pandas, scikit-learn — do their work in compiled C
code that releases the GIL, so many threads in one process genuinely run in parallel,
and they share their data without copying it. Code written in Python cannot do that:
the GIL lets only one thread run at a time, so parallelism has to come from separate
processes. Mostly-NumPy work therefore wants few processes with many threads each,
mostly-Python work wants many processes with one or two threads, and a mixture wants
something in between. The values below lean towards the numerical case.

In [ ]:
# ---------------------------------------------------------------------------
# Site configuration. On a cluster other than the bwUniCluster, every line in
# this block is a candidate for change.
# ---------------------------------------------------------------------------
queue      = "cpu"           # partition; "cpu" allows up to 20 nodes per job
mpi_module = "mpi/openmpi"   # module providing mpirun and libmpi
interface  = "ib0"           # interface for worker-to-worker traffic (Infiniband)

cores_per_node = 96          # a "cpu" node on bwUniCluster 3.0
mem            = "360000mb"  # a little under the 384 GB the node actually has

# ---------------------------------------------------------------------------
# Job size.
#   For the "multi" queue count_nodes must be at least 2, for "single" exactly 1.
# ---------------------------------------------------------------------------
count_nodes           = 2
count_worker_per_node = 2
walltime              = "30:00"

# Two ranks per node over 96 cores -> 48 threads each, and every core is used.
count_threads_per_worker = cores_per_node // count_worker_per_node

count_ranks   = count_nodes * count_worker_per_node
count_workers = count_ranks - 1        # one rank becomes the scheduler

print(f"{count_nodes} nodes x {count_worker_per_node} ranks per node"
      f" x {count_threads_per_worker} threads per rank")
print(f"{count_ranks} MPI ranks -> 1 scheduler + {count_workers} workers")
print(f"{count_worker_per_node * count_threads_per_worker}"
      f" of {cores_per_node} cores used per node")

Check that last line before submitting. If the cores used per node is well below the
cores the node has, the job is reserving hardware it will not touch — and if the
product of ranks and threads *exceeds* the cores, the threads will fight each other
for them.

<a id="jobscript"></a>
## 4. The job script

`sbatch` reserves the nodes and then runs one script on the first of them. That script
has to do three things in order: make the environment available on every node, work out
which nodes it got, and start `dask-mpi` across them with `mpirun`.

Two lines in it are easy to get wrong and worth reading carefully.

**`module load` is not optional.** `mpirun` comes from the cluster's MPI module, and so
does the `libmpi` that `mpi4py` needs. The `mpi4py` wheel from PyPI does not bundle an
MPI of its own — it loads the system one when it is imported. If the module is missing,
every rank ends up in an MPI world of size one, and `dask-mpi` fails with
`WorldTooSmallException: ... found 1 MPI ranks but needs 2`, once per rank. That
message always means the MPI that launched the processes and the MPI that `mpi4py`
found are not the same one.

**`--map-by ppr:N:node` is what puts ranks on different nodes.** Asking instead for
`core:PE=<threads>` only distributes ranks as a side effect of the first node running
out of cores — so with 4 ranks of 48 threads on 96-core nodes, all four fit on node one
and the second node sits idle for the whole job. Nothing fails, the dashboard looks
healthy, and the cluster is silently not a cluster. `ppr:N:node` states the placement
instead of hoping for it.

The script is generated by a function, because [section 8](#growing) needs the same
script again with one flag changed.

In [ ]:
scheduler_file = pathlib.Path.home() / "dask-scheduler.json"


def dask_mpi_job_script(worker_name, with_scheduler=True):
    """Return a SLURM job script that starts dask-mpi on the reserved nodes.

    with_scheduler=False starts workers only. They join the cluster described by
    scheduler_file instead of creating a new one, which is how an existing cluster
    is enlarged (section 8).
    """
    no_scheduler = "" if with_scheduler else "--no-scheduler"

    # rf'''  -- a raw string, so that the trailing backslashes stay in the
    # generated script as bash line continuations instead of being consumed by Python.
    return rf'''#!/bin/bash -l
# -l makes this a login shell, so that "module" is defined.

# mpirun and libmpi both come from the cluster's MPI module. The mpi4py wheel
# loads the system libmpi at import time, so this has to happen before any
# worker starts.
module load {mpi_module}

# uv reads its package cache from $HOME, which was filled on the login node, and
# writes the environment to the node's local disk.
export UV_CACHE_DIR="$HOME/.cache/uv"
export UV_LINK_MODE=copy    # cache and $TMP are on different filesystems
export UV_OFFLINE=1         # compute nodes cannot reach PyPI

# $TMP is a directory on the local disk of each node, unique to this job.
VENV="$TMP/envs/venv"
DASK_DIR="$TMP/dask"

# Empty for the first job; "--no-scheduler" for a job that only adds workers to an
# existing cluster. Unquoted below, so an empty value adds no argument at all.
NO_SCHEDULER="{no_scheduler}"

DASK_HOSTS=""
for NODE in $(scontrol show hostname); do
    DASK_HOSTS+="${{NODE}},"

    # One srun per node, all started in parallel thanks to the trailing "&".
    # Each builds the environment on the local disk of its own node.
    srun -N 1 -n 1 -w "$NODE" /bin/bash -c "
        rm -rf '$TMP/envs' '$DASK_DIR' &&
        mkdir -p '$TMP/envs' '$DASK_DIR' &&
        cd '{repo_root}' &&
        UV_PROJECT_ENVIRONMENT='$VENV' \
            {uv_bin} sync --frozen --no-dev --extra dask --extra mpi
    " &
done

wait                              # for all the sruns started above
DASK_HOSTS="${{DASK_HOSTS%,}}"    # drop the trailing comma

echo "environment ready on: $DASK_HOSTS"

# -x PATH -x LD_LIBRARY_PATH  hands the module's environment to the remote ranks,
#     which never read your shell profile and so cannot find libmpi on their own.
# --map-by ppr:N:node  places exactly N ranks on each node (see above).
# --worker-class distributed.Worker  runs workers without a supervising nanny.
mpirun -x PATH -x LD_LIBRARY_PATH \
    --host "$DASK_HOSTS" \
    --map-by ppr:{count_worker_per_node}:node:PE={count_threads_per_worker} \
    -np {count_ranks} \
    "$VENV/bin/dask-mpi" \
        --scheduler-file '{scheduler_file}' \
        --interface {interface} \
        --local-directory "$DASK_DIR" \
        --worker-class distributed.Worker \
        --nthreads {count_threads_per_worker} \
        $NO_SCHEDULER \
        --name {worker_name} \
        --dashboard-address :8787
'''


job_file = pathlib.Path.home() / "job_dask_mpi.sh"
job_file.write_text(dask_mpi_job_script("base"))

print(job_file.read_text())

Read the generated script rather than trusting it — every value from
[section 3](#sizing) has been substituted in, and this is the last point at which a
wrong number is cheap to fix.

Two details in there repay a second look. `DASK_HOSTS` is built up inside the loop and
has its trailing comma removed afterwards, because `mpirun` rejects a host list with an
empty entry at the end. And `--local-directory` points into `$TMP`: when a worker runs
out of memory it spills to disk, and that spilling should go to the node's own disk
rather than across the network to `$HOME`.

You may find `--host` unnecessary. Open MPI built with SLURM support reads the node
list from the allocation by itself, and dropping the flag is one fewer thing to get
wrong. It is kept here because it also works in queues that do not export the list.

<a id="submitting"></a>
## 5. Submitting and monitoring the job

How many nodes are free right now, per queue. `sinfo_t_idle` is a bwHPC helper script,
not part of SLURM, and it is also available in a terminal
(*File* &rarr; *New* &rarr; *Terminal*).

In [ ]:
print(shell("sinfo_t_idle"))

Submitting means deleting any scheduler file left over from an earlier run — its
reappearance is how we detect that the cluster is up — and then handing the script to
`sbatch` with the resources from [section 3](#sizing).

`sbatch` replies with the job id, and every later step needs it: to read the job's
output, to wait for it, and to cancel it at the end. So the reply is parsed and kept
rather than copied by hand.

In [ ]:
submitted_jobs = []


def submit(script_path):
    """Submit a job script, remember its id, and return it."""
    output = shell(
        f"sbatch"
        f" -p {queue}"
        f" --nodes={count_nodes}"
        f" --ntasks={count_ranks}"
        f" --ntasks-per-node={count_worker_per_node}"
        f" --time={walltime}"
        f" --mem={mem}"
        f" {script_path}"
    )
    print(output)

    match = re.search(r"Submitted batch job (\d+)", output)
    if not match:
        raise RuntimeError(f"sbatch returned no job id:\n{output}")

    submitted_jobs.append(match.group(1))
    return match.group(1)


scheduler_file.unlink(missing_ok=True)

job_id = submit(job_file)
slurm_log = pathlib.Path.cwd() / f"slurm-{job_id}.out"

print("job id:  ", job_id)
print("log file:", slurm_log)

While the job waits for its nodes, `squeue` shows what it is doing. The `ST` column is
the interesting one: `PD` is pending, `R` is running, `CG` is finishing. The full list
of codes is at
[https://curc.readthedocs.io/en/latest/running-jobs/squeue-status-codes.html](https://curc.readthedocs.io/en/latest/running-jobs/squeue-status-codes.html).

`NODELIST` names the nodes with the numbers collapsed into brackets: `uc3n[001-003]`
means three consecutive nodes, `uc3n[001,003]` means two separate ones.

In [ ]:
print(shell("squeue --me"))

In [ ]:
print(shell("squeue --me --start"))      # an estimated start time, if SLURM has one

Now wait for the cluster. The scheduler writes `~/dask-scheduler.json` once the workers
have registered, so the file appearing is the signal that everything is ready.

The waiting loop has a deadline, which matters more than it might seem: if the job dies
during startup — a missing module, a failed `uv sync` — the file never appears, and a
loop without a timeout would simply hang with nothing on screen. On expiry this one
prints the job's state and the end of its log, which is where the reason will be.

In [ ]:
timeout_minutes = 20
deadline = time.time() + timeout_minutes * 60

while not scheduler_file.exists():
    if time.time() > deadline:
        print(f"No scheduler file after {timeout_minutes} minutes.\n")
        print(shell(f"squeue --me --job {job_id}"))
        if slurm_log.exists():
            print(f"\n--- end of {slurm_log.name} ---")
            print(shell(f"tail -n 40 {slurm_log}"))
        else:
            print(f"\n{slurm_log.name} does not exist yet"
                  f" -- the job is most likely still queued.")
        raise TimeoutError("the Dask cluster did not start; see the output above")
    time.sleep(2)

print(f"{scheduler_file} written -- the cluster is up")

The job's own output is worth a look either way. `dask-mpi` logs the address the
scheduler is listening on and one line per worker that registers.

In [ ]:
print(shell(f"cat {slurm_log}"))

<a id="connecting"></a>
## 6. Connecting to the cluster

The scheduler file holds the scheduler's address, so `Client` needs nothing else. This
is the only line in the notebook that differs from the first notebook's `Client()`, and
everything after it is identical.

`wait_for_workers` blocks until the expected number of workers has registered.
Without it the next cell might run on a cluster that is still only half assembled, and
appear mysteriously slow.

In [ ]:
from dask.distributed import Client

client = Client(scheduler_file=str(scheduler_file))
client.wait_for_workers(count_workers)

client

Check the printout against [section 3](#sizing): the number of workers, the threads per
worker, and the total memory should all be what you asked for. A worker count that is
right but a *node* count that is not is the symptom of the placement problem described
in [section 4](#jobscript) — the dashboard's *Workers* tab shows which host each worker
is on.

Note that no `temporary_directory` is configured here. The workers already received
`--local-directory` in the job script, and that is where spilling has to be set,
because it is the workers that spill — not this notebook.

### 6.1 The dashboard

Forwarding a port to reach the dashboard works exactly as in
[section 2.1 of the first notebook](01_Introduction_To_Dask.ipynb#client), with one
difference: the host in the URL is now the node running the **scheduler**, not the one
running Jupyter.

```bash
ssh -N -L 8787:<scheduler-node>:8787 <university abbreviation>_<User-ID>@bwunicluster.scc.kit.edu
```

The scheduler's address is in the client printout above and in the job log. With the
tunnel open the dashboard is at
[http://localhost:8787/status](http://localhost:8787/status), and it is genuinely worth
having open for the rest of this notebook — on several nodes it is the only practical
way to see whether work is actually being spread out.

The `dask-labextension` plugin, which embeds the dashboard in JupyterLab, does not
work with `dask-mpi` here; the tunnel is the way.

<a id="computing"></a>
## 7. Computing at cluster scale

Nothing in this section is new API. It is the code from the first notebook, on data
sized for the hardware you have just reserved.

### 7.1 An array that needs several nodes

In [ ]:
import dask.array as da

x = da.random.random((100000, 100000), chunks="128 MiB")

print("size in GB:", round(x.nbytes / 1e9, 1))
print("chunks:    ", x.numblocks)
x

80 GB — more than one node's memory, and spread across all of them. The calculation
below is chosen to be awkward on purpose: `x.T` means every chunk needs a chunk from
somewhere else, so this is not the embarrassingly parallel case but a real shuffle
between workers, and between nodes over Infiniband.

Watch the dashboard while it runs. The *Task Stream* shows red for inter-worker
transfers, and there will be plenty.

In [ ]:
%%time

y = (x + x.T) - x.mean(axis=0)
y.sum().compute()

### 7.2 A DataFrame read by the workers

The path has to be **absolute**. The workers are processes on other nodes with their
own working directories, and a relative path would be resolved against whichever
directory SLURM happened to give them.

In [ ]:
import dask.dataframe as dd

taxi_file = repo_root / "files" / "green_tripdata_2023-01.parquet"

taxi = dd.read_parquet(taxi_file, engine="pyarrow")

print("file:      ", taxi_file)
print("partitions:", taxi.npartitions)
taxi.head(3)

This is the workshop's teaching file, about 1.4 MB — far too small to need a cluster,
and slower here than plain pandas would be, for the reasons the first notebook measured
in its section 4.8. Use it to check that the machinery works. The line that would make
the reservation worthwhile is the same one with a pattern in place of a filename:

```python
taxi = dd.read_parquet('/pfs/work/<your-workspace>/taxi/green_tripdata_*.parquet')
```

`persist()` becomes considerably more interesting on a cluster than it was locally,
because "worker memory" is now the combined memory of every node — several hundred GB
here, against the few GB a single process can hold.

In [ ]:
from dask.distributed import wait

taxi = taxi.persist()
_ = wait(taxi)

print("rows:", len(taxi))

### 7.3 Functions as tasks: `delayed`

Arrays and DataFrames cover tabular and numerical work, but an ordinary Python function
can also become a task. The `delayed` decorator makes a function lazy: calling it
records a task instead of running the function.

The example below is deliberately built the wrong way, because the mistake it makes is
the most common one on a cluster.

In [ ]:
import dask


@dask.delayed
def increment(x):
    return x + 1


@dask.delayed
def square(x):
    return x ** 2


@dask.delayed
def add(x, y):
    return x + y


numbers = list(range(1, 10000))

# Three tasks per number, ~30000 in total, each doing one arithmetic operation.
total_slow = dask.delayed(sum)([add(increment(n), square(n)) for n in numbers])
total_slow

In [ ]:
%%time

total_slow.compute()

Slow — quite possibly slower than a plain Python loop, and Dask may well have printed a
warning about the size of the graph. Every task costs the scheduler roughly a
millisecond to hand out and track, whatever it does, and here each task does a single
addition. All the time goes into organising the work rather than doing it.

The graph shows why. The same construction over five numbers instead of ten thousand is
readable; see [section 3.3 of the first
notebook](01_Introduction_To_Dask.ipynb#array) for what the shapes mean.

In [ ]:
dask.delayed(sum)([add(increment(n), square(n)) for n in range(1, 6)]).visualize(rankdir="LR")

Multiply that by two thousand and the problem is obvious. The fix is not fewer
operations but **bigger tasks**: give each one enough work to be worth a millisecond of
scheduling. Here that means handing every task a block of numbers rather than one
number.

In [ ]:
@dask.delayed
def process_block(block):
    return sum((n + 1) + (n ** 2) for n in block)


blocks = [numbers[i:i + 1000] for i in range(0, len(numbers), 1000)]

total_fast = dask.delayed(sum)([process_block(b) for b in blocks])

print("tasks before:", 3 * len(numbers) + 1)
print("tasks now:   ", len(blocks) + 1)

In [ ]:
%%time

total_fast.compute()

The same answer from ten tasks instead of thirty thousand, and much faster.

In [ ]:
print("identical result:", total_slow.compute() == total_fast.compute())

This is the same argument as the chunk size in
[section 3.4 of the first notebook](01_Introduction_To_Dask.ipynb#array), and it applies
to every Dask collection. Note also that `delayed` functions cannot be used in an `if`
or as a loop condition: the value is not known when the graph is built. The API is
documented at
[https://docs.dask.org/en/stable/delayed-api.html](https://docs.dask.org/en/stable/delayed-api.html).

<a id="growing"></a>
## 8. Growing the cluster while it runs

Workers do not all have to come from the same SLURM job. A second job can start more of
them and point them at the scheduler that is already running, and they join the existing
cluster — which is useful when a job is already under way and turns out to need more
resources than were reserved for it.

The only change to the job script is `--no-scheduler`, which is why
[section 4](#jobscript) generated it from a function. The new workers find the scheduler
through the same `~/dask-scheduler.json`.

In [ ]:
expand_file = pathlib.Path.home() / "job_dask_mpi_expand.sh"
expand_file.write_text(dask_mpi_job_script("expansion", with_scheduler=False))

job_id_expand = submit(expand_file)
print("expansion job id:", job_id_expand)

Every rank of the second job becomes a worker, since none of them is taken by a
scheduler. So the cluster grows from `count_workers` to `count_workers + count_ranks`.

In [ ]:
expected = count_workers + count_ranks

print("waiting for", expected, "workers ...")
client.wait_for_workers(expected)

client

The *Workers* tab of the dashboard now lists workers with two different name prefixes,
`base-` and `expansion-`, on nodes from two different SLURM jobs. Dask itself makes no
distinction: the scheduler simply has more workers to give tasks to, and any
computation started from here uses all of them.

Both jobs hold their nodes until they are cancelled — [section 10](#freeing) deals with
that.

<a id="ml"></a>
## 9. Machine learning on the cluster

Chapter 06 trained models on data that fitted in memory. The same pipeline can be run
across the cluster, which is worth doing when either the data or the search over
hyperparameters is too large for one node.

The features come from the taxi data: the question is whether a trip received a large
tip, defined as more than 20% of the fare.

*Based on [https://github.com/rikturr/high-performance-jupyter](https://github.com/rikturr/high-performance-jupyter) (MIT licence).*

### 9.1 Preparing the features

In [ ]:
numeric_feat = [
    'pickup_weekday',
    'pickup_hour',
    'pickup_week_hour',
    'pickup_minute',
    'passenger_count',
]
categorical_feat = [
    'PULocationID',
    'DOLocationID',
]
features = numeric_feat + categorical_feat
y_col = 'high_tip'


def prep_df(df):
    """Build the model features from the raw taxi columns.

    Runs on the Dask DataFrame as a whole; Dask applies it partition by partition.
    """
    df = df[df.fare_amount > 0]                      # avoids dividing by zero below
    df['tip_fraction'] = df.tip_amount / df.fare_amount
    df[y_col] = df['tip_fraction'] > 0.2

    df['pickup_weekday'] = df.lpep_pickup_datetime.dt.weekday
    df['pickup_hour'] = df.lpep_pickup_datetime.dt.hour
    df['pickup_week_hour'] = (df.pickup_weekday * 24) + df.pickup_hour
    df['pickup_minute'] = df.lpep_pickup_datetime.dt.minute

    # Everything the model sees becomes a float; missing values become -1.
    return df[features + [y_col]].astype(float).fillna(-1)


taxi_ml = prep_df(taxi).persist()
_ = wait(taxi_ml)

taxi_ml.head(3)

In [ ]:
seed = 42

taxi_sample = taxi_ml.sample(frac=0.02, replace=False, random_state=seed).persist()
_ = wait(taxi_sample)

print("rows in the sample:", len(taxi_sample))

### 9.2 A grid search with dask_ml

`dask_ml` mirrors parts of the scikit-learn API with versions that accept Dask
collections, so the data never has to be gathered into one process. The pipeline
below categorises the two location columns, one-hot encodes them, scales the numeric
columns, and fits a logistic regression — then searches over `l1_ratio` with
three-fold cross-validation.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from dask_ml.compose import ColumnTransformer
from dask_ml.preprocessing import StandardScaler, DummyEncoder, Categorizer
from dask_ml.model_selection import GridSearchCV

lr = LogisticRegression(
    solver='saga',
    penalty='elasticnet',
    l1_ratio=0.5,
    max_iter=100,
    random_state=seed,
)

pipeline = Pipeline(steps=[
    ('categorize', Categorizer(columns=categorical_feat)),
    ('onehot', DummyEncoder(columns=categorical_feat)),
    ('scale', ColumnTransformer(
        transformers=[('num', StandardScaler(), numeric_feat)])),
    ('clf', lr),
])

grid_search = GridSearchCV(
    pipeline,
    {'clf__l1_ratio': [0.2, 0.3, 0.5, 0.7, 0.9]},
    cv=3,
    scoring='accuracy',
)

In [ ]:
%%time

grid_search.fit(taxi_sample[features], taxi_sample[y_col])

print("best score: ", round(grid_search.best_score_, 4))
print("best params:", grid_search.best_params_)

### 9.3 If that cell failed

`dask_ml` is released far less often than `dask` itself, and a version combination in
which its preprocessing steps no longer match the installed pandas or scikit-learn is
entirely possible. **Run this section early** so that a failure here does not surprise
you at the end of a workshop.

If it did fail, the fallback below gives up on Dask collections and uses plain
scikit-learn on a pandas DataFrame — but hands the cross-validation out to the Dask
cluster through joblib, which scikit-learn uses internally for parallelism. The data
has to fit in one process, which our sample comfortably does; the grid search still
runs on every node.

In [ ]:
import joblib
from sklearn.compose import ColumnTransformer as SkColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler as SkStandardScaler
from sklearn.model_selection import GridSearchCV as SkGridSearchCV

sample_pdf = taxi_sample.compute()      # a pandas DataFrame, on this node

sk_pipeline = Pipeline(steps=[
    ('prep', SkColumnTransformer(transformers=[
        ('num', SkStandardScaler(), numeric_feat),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_feat),
    ])),
    ('clf', LogisticRegression(solver='saga', penalty='elasticnet',
                               l1_ratio=0.5, max_iter=100, random_state=seed)),
])

sk_search = SkGridSearchCV(
    sk_pipeline,
    {'clf__l1_ratio': [0.2, 0.3, 0.5, 0.7, 0.9]},
    cv=3,
    scoring='accuracy',
)

with joblib.parallel_backend('dask'):
    sk_search.fit(sample_pdf[features], sample_pdf[y_col])

print("best score: ", round(sk_search.best_score_, 4))
print("best params:", sk_search.best_params_)

<a id="freeing"></a>
## 10. Freeing the resources

This section is not optional. A SLURM job holds its nodes until it finishes or is
cancelled, and a 30-minute reservation that nobody uses is 30 minutes of two nodes
taken away from everyone else on the machine.

Closing the client disconnects this notebook; it does not end the jobs. Both have to be
cancelled explicitly, which is what the ids collected by `submit` are for.

In [ ]:
client.close()

for jid in submitted_jobs:
    print(shell(f"scancel {jid}"), f"<- cancelled {jid}")

In [ ]:
print(shell("squeue --me") or "no jobs left in the queue")

---

That is the whole of it. Reserve nodes with one SLURM job, build the environment on
each node's local disk with `uv`, start scheduler and workers across them with
`dask-mpi`, connect a `Client` through the scheduler file — and from that point on the
Dask code is the code from the first notebook, unchanged.

Further reading:

- Dask on HPC: [https://docs.dask.org/en/stable/deploying-hpc.html](https://docs.dask.org/en/stable/deploying-hpc.html)
- `dask-mpi`: [https://mpi.dask.org/](https://mpi.dask.org/)
- bwUniCluster queues: [https://wiki.bwhpc.de/e/BwUniCluster3.0/Running_Jobs](https://wiki.bwhpc.de/e/BwUniCluster3.0/Running_Jobs)
- bwHPC Wiki on Python: [https://wiki.bwhpc.de/e/Python](https://wiki.bwhpc.de/e/Python)